# FinTrace 360: PaySim Mobile Money Transaction Analytics
**Module:** Secondary Dataset Ingestion & Mobile Payment Fraud Profiling  
**Reference:** Synthetic Mobile Money Benchmark (Lopez-Rojas et al.)  
**Objective:** Analyze mobile transfer volumes, examine balance discrepancy anomalies, and investigate TRANSFER vs CASH_OUT fraud concentration.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE_DIR))

from src.ingestion.paysim_loader import load_paysim

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.autolayout": True, "figure.dpi": 150})


## 1. Load PaySim Dataset with Memory Optimization


In [ ]:
df_paysim = load_paysim()
print(f"PaySim Shape: {df_paysim.shape}")
print(f"Memory Usage: {df_paysim.memory_usage(deep=True).sum() / (1024*1024):.2f} MB")
df_paysim.head()


## 2. Transaction Types & Fraud Concentration Analysis


In [ ]:
type_summary = df_paysim.groupby("type", observed=False).agg(
    total_tx=("amount", "count"),
    total_volume=("amount", "sum"),
    fraud_count=("isFraud", "sum"),
    fraud_rate_pct=("isFraud", lambda x: x.mean() * 100)
).reset_index()
display(type_summary)


## 3. Fraud Exclusivity in TRANSFER and CASH_OUT


In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=type_summary, x="type", y="fraud_count", palette="Reds_r")
plt.title("Distribution of Confirmed Fraud by Transaction Rail")
plt.xlabel("Transaction Type")
plt.ylabel("Fraud Cases")
plt.show()


## 4. Origin Balance Discrepancy Feature


In [ ]:
# In legitimate transactions: oldbalanceOrg - amount == newbalanceOrig
# In fraudulent wipes: discrepancy indicates anomalous account drain
df_paysim['orig_discrepancy'] = df_paysim['newbalanceOrig'] + df_paysim['amount'] - df_paysim['oldbalanceOrg']
print("Mean Discrepancy for Benign Transactions:", df_paysim[df_paysim['isFraud'] == 0]['orig_discrepancy'].mean())
print("Mean Discrepancy for Fraudulent Transactions:", df_paysim[df_paysim['isFraud'] == 1]['orig_discrepancy'].mean())
